## Diagrams to Illustrate Dreadnought Self-Destruct

In [15]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:70% !important; }</style>"))

In [67]:
!mkdir -p selfdestruct/trigrams
!mkdir -p selfdestruct/trigrams_simple
!mkdir -p selfdestruct/trigrams_labelled

In [48]:
from PIL import Image, ImageDraw, ImageColor, ImageFont
for LEVEL in range(1,16):
    for a in ['F9','FA']:
        for b in ['FB','FC']:
            for c in ['FD','FE']:
                for padding in range(0,20):
                    img = Image.new('RGBA', ((120 * 3) + (120 * padding),120))
                    draw = ImageDraw.Draw(img)
                    X=0
                    for x in (a,b,c):
                        cimg = Image.open(f"surface_characters/{LEVEL}_{x}.png")
                        img.paste(cimg,(X,0))
                        draw.rectangle((X+1, 1, X+120-1, 120-1), fill=None, outline="red",width=2)
                        X+=120
                    for x in range(0, padding):
                        draw.rectangle((X+1, 1, X+120-1, 120-1), fill=None, outline="red",width=2)
                        X+=120
                    img.save(f"selfdestruct/trigrams/{LEVEL}_{a}_{b}_{c}_{padding}.png")

                    img = Image.new('RGBA', ((120 * 3) + (120 * padding),120))
                    draw = ImageDraw.Draw(img)
                    X=0
                    for x in (a,b,c):
                        cimg = Image.open(f"surface_characters/{LEVEL}_{x}.png")
                        img.paste(cimg,(X,0))
                        draw.rectangle((X, 0, X+120, 120), fill=None, outline="black",width=1)
                        X+=120
                    for x in range(0, padding):
                        draw.rectangle((X, 0, X+120, 120), fill=None, outline="black",width=1)
                        X+=120
                    img.save(f"selfdestruct/trigrams/{LEVEL}_{a}_{b}_{c}_{padding}_no_highlight.png")
                    
                X=0
                img = Image.new('RGBA', ((145 * 3)+10, 215))
                draw = ImageDraw.Draw(img)
                for x in (a,b,c):
                    cimg = Image.open(f"character_mini_diagrams/{LEVEL}_{x}.png")
                    img.paste(cimg,(X,0))
                    X+=145
                img.save(f"selfdestruct/trigrams_labelled/{LEVEL}_{a}_{b}_{c}.png")


### Progressive draw of screen

In [3]:
from PIL import Image, ImageDraw, ImageColor, ImageFont
def generateSurfaceDiagramWithGrid(level_name, level_number, level_image, description, X_START, X_END):
    img = Image.new('RGBA', (50 + level_image.width + 140, 230 + level_image.height + 20))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(0,0),img.size], fill = "white")

    # Sprite label
    #label_text = f"URIDIUM/{('00'+str(level_number))[-2:]}/{level_name}/{description}"
    label_text = f"{level_name}/{description}"
    label_fnt_size = 88
    label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size + 10
    txt = Image.new('RGBA', (txt_width, label_fnt_size))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    img.paste(txt, (140, 5))

    # Grid Literals - Rows
    grid_literals = []
    for n in range(0x82, 0xA4, 2):
        grid_literals += [hex(n)[2:].upper()]
    label_text = '\n'.join(grid_literals)
    label_fnt_size = 113
    label_fnt = ImageFont.truetype("JetBrainsMono-Regular.ttf", label_fnt_size)
    txt_height = len(label_text) * label_fnt_size
    txt_width =  (label_fnt_size + 25)
    txt = Image.new('RGBA', (txt_width, txt_height))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="gray")
    label = txt
    img.paste(label, (10,80),mask=label)

    # Grid Literals - Cols
    grid_literals = []
    for n in range(X_START, X_END, 1):
        grid_literals += [hex(n)[2:].upper()]
    label_fnt_size = 113
    label_fnt = ImageFont.truetype("JetBrainsMono-Regular.ttf", label_fnt_size)
    x = 150
    for label_text in grid_literals:
        txt_height = (label_fnt_size + 25)
        txt_width =  len(label_text) * label_fnt_size
        txt = Image.new('RGBA', (txt_width, txt_height))
        draw = ImageDraw.Draw(txt)
        draw.rectangle([(0,0), txt.size], fill = "white")
        draw.text((0, 0), label_text, font=label_fnt, fill="gray")
        label = txt.rotate(90,expand=1)
        img.paste(label,(x,(img.height - label.height)),mask=label)
        x += 120

    # Main Sprite image
    img.paste(level_image, (160,100),mask=level_image)

    return img


In [53]:
from PIL import Image
import random

half_width = 11040
LEVEL = 11
SEGMENT = 6
X_START = 0x20 * SEGMENT
X_END = 0x20 * (SEGMENT+1)
crop_start = X_START * 8 * 15
crop_end = X_END * 8 * 15

offset = 0
offsets = []
for i in range(0,17):
    step = random.randint(-1,1)
    offset = max(0, min(offset + step, 4))
    offsets += [offset]
print(offsets)
for pic_type in ["", "_no_highlight"]:
    img = Image.open(f"scrolling/surface_diagrams/{LEVEL}_no_text.png")
    img1 = img.crop((crop_start, 0, crop_end, img.height))
    img1 = generateSurfaceDiagramWithGrid("TRI-ALLOY", LEVEL, img1, f"SELF/DESTRUCT", X_START, X_END)
    padding = 3
    for x in range(27,17,-1):
        for y in range(0,17):
            offset = x + offsets[y]
            padding = 32 - offset - 3
            a,b,c = random.choice(['F9','FA']),random.choice(['FB','FC']),random.choice(['FD','FE'])
            cimg = Image.open(f"selfdestruct/trigrams/{LEVEL}_{a}_{b}_{c}_{padding}{pic_type}.png")
            img1.paste(cimg, (160 + (offset * 120), 100 + (y * 120)))
        img1.save(f"selfdestruct/sequence/{LEVEL}_destruct_{x}{pic_type}.png")
        img2 = img1.crop((0, 100 + (7 * 120), img1.width, 100 + (10 * 120)))
        img2.save(f"selfdestruct/sequence/{LEVEL}_destruct_{x}_clip{pic_type}.png")
    img2


[0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1]
